# Rec Centre & Bike Share: Your First Charts
SDEV2501 · Day 13 · **Student notebook**

For three lessons we've been scrubbing the September bookings until they squeaked. Today we finally get to look at them. The rule for the day: **a chart answers a question.** If you can't say the question out loud, you're not ready to pick a chart yet.

We'll use two files you already know:

- `bookings_sept_clean.csv`: the September bookings you cleaned on Day 10 (140 rows), plus `facilities.csv`.
- `bike_share_mock.csv`: 400 mock bike-share rides from May and June (Day 8).

Every chart today goes through pandas' `.plot()`, which is matplotlib underneath. You'll see `fig, ax = plt.subplots()` a lot. `fig` is the whole image; `ax` (an *axes*) is one chart inside it, and it's the thing you call `.set_title()` and friends on.

## 0. Setup
Run this cell. It loads both files, adds a `booking_total` column, and joins in the facility names.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# September bookings (already cleaned on Day 10) + facility names
bookings = pd.read_csv("bookings_sept_clean.csv", parse_dates=["booking_date"])
facilities = pd.read_csv("facilities.csv")
bookings["booking_total"] = bookings["hours"] * bookings["hourly_rate"]
bookings = bookings.merge(facilities, on="facility_id", how="left")
bookings["facility_name"] = bookings["facility_name"].fillna("Unknown (F07)")

# Bike-share rides
rides = pd.read_csv("bike_share_mock.csv", parse_dates=["start_time"])

print(bookings.shape, rides.shape)

## 1. Question first, chart second
Before any code, here's the cheat table we'll use all day. The left column is the question; the middle is what kind of data answers it.

| The question sounds like... | The data is... | Reach for |
|---|---|---|
| "Which is biggest / how do they rank?" | one number per category | **bar chart** (sorted) |
| "What share of the whole is this?" | parts that add up to 100% | bar chart; a **pie** only for 2-3 slices |
| "What does the spread of values look like?" | one continuous column | **histogram** |
| "How do the spreads of several groups compare?" | continuous column + a category | **boxplot** (or several histograms) |
| "How did it change over time?" | a number per time step | **line chart** |
| "Are these two numbers related?" | two continuous columns | **scatter plot** |
| "I need the exact values" | a handful of numbers | **a table** (slide: you don't always need a graphic) |

*Continuous* data can take any value in a range (ride duration: 11.3 min). *Discrete* data comes in separate chunks (activity, station, number of bookings). That one distinction rules out half the options before you start.

## 2. Bar chart: which activity gets booked most?
One number per category, so it's a bar chart. Count bookings per activity, sort them, and draw horizontal bars so the labels are readable without tilting your head.

**Your turn:** make a horizontal bar chart of bookings per activity, with the biggest at the top, a title, and an x-axis label.

You'll need: `value_counts()`, `sort_values()`, `.plot(kind="barh", ax=ax)`, `ax.set_title()`, `ax.set_xlabel()`.

## 3. Same data as a pie (principle #5)
**Your turn:** draw the same counts as a pie chart, then answer the question in the Markdown cell below *using only the pie*.

You'll need: `.plot(kind="pie", ax=ax)`, `ax.set_title()`.

**Write it down:** from the pie alone, which is bigger: Gym Drop-In, Court Rental or Public Skate?

*Your answer:*

## 4. Histogram: how long is a typical bike ride?
Ride duration is continuous, so we want its **distribution**: how many rides fall in each range of minutes. That's a histogram. Bars touch (no gaps) because the ranges, called *bins*, run into each other.

**Your turn:** draw a histogram of `duration_min` with 30 bins. Then print `describe()` for the same column.

You'll need: `.plot(kind="hist", bins=..., ax=ax)`, `describe()`.

That chart is mostly empty. A few rides of 3-6 hours stretch the x-axis so far that every normal ride gets squashed into the first two bars. The chart just told you something the mean (18.3 min) hid: there's a long tail of extreme rides.

**Your turn:** redraw the histogram for rides of 60 minutes or less, and add a vertical line at the median.

You'll need: a boolean mask with `.loc`, `median()`, `ax.axvline()`.

## 5. Boxplot: how does booking value differ by activity?
Now we want to compare a continuous number (`booking_total`) **across groups** (activity). A boxplot draws five numbers per group: the box is the middle 50% (Q1 to Q3), the line inside is the median, the whiskers reach to the last value within 1.5 × IQR of the box, and dots beyond that are flagged as outliers. Same IQR rule as Day 11.

**Your turn:** make a boxplot of `booking_total` grouped by `activity`, using every booking that wasn't cancelled.

You'll need: a boolean mask on `status`, `.boxplot(column=..., by=..., ax=ax)`, and `fig.suptitle("")` (pandas adds an automatic title you'll want to delete).

**Write it down:** one dot sits way above everything else. Which booking is it, and is it a real booking? (Hint: look at the table under the chart and remember Day 10.)

*Your answer:*

## 6. Line chart: are rides going up or down?
A number per time step means a line chart. `resample("W")` groups a datetime column into weeks, the same way `groupby` groups a category.

**Your turn:** count rides per week and plot them as a line.

You'll need: `set_index("start_time")`, `resample("W")`, `size()`, `.plot(ax=ax)`.

**Write it down:** the first and last points are much lower than the rest. Did ridership really crash twice?

*Your answer:*

## 7. Scatter plots and small multiples: Anscombe's quartet
Slide 30 showed four datasets with the same mean, variance, correlation and regression line. Let's check that claim ourselves, then plot them. Four related charts in a grid with shared axes is called **small multiples** (slides 43-46).

Run the next cell to build the data (it's the real Anscombe 1973 data, typed in).

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = pd.DataFrame({
    "set": ["I"] * 11 + ["II"] * 11 + ["III"] * 11 + ["IV"] * 11,
    "x": x123 * 3 + [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
    "y": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68,
          9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74,
          7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73,
          6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
})

**Your turn:** for each set, show the mean of x, mean of y, standard deviation of y, and the x-y correlation.

You'll need: a `for` loop over `groupby("set")`, `.mean()`, `.std()`, `.corr()`.

Four identical rows. Now plot them.

**Your turn:** make a 2 × 2 grid of scatter plots, one per set, sharing both axes.

You'll need: `plt.subplots(2, 2, sharex=True, sharey=True)`, `axes.flat`, `ax.scatter()`.

## 8. Exploratory → explanatory: remove to improve
Everything so far was **exploratory**: charts for *you*, fast and ugly, to find out what's in the data. An **explanatory** chart is for someone else. It has one message and everything that doesn't serve that message goes.

Question: *which facility made the most money in September?* Use completed bookings that don't need review (the Day 10 rule).

**Your turn (exploratory):** make the quickest possible bar chart of revenue per facility. Don't style it.

You'll need: a boolean mask on `status` and `needs_review`, `groupby("facility_name")`, `sum()`, `.plot(kind="bar")`.

That's fine for you. Now make it for your manager. Work through this list one change at a time and re-run after each:

1. Sort it, and make it horizontal so the names are readable.
2. Make every bar grey except the one the message is about.
3. Put the values on the bars and delete the x-axis (the labels replace it).
4. Remove the box (the "spines") around the chart.
5. Replace the title with the **finding**, not the topic.

You'll need: `sort_values()`, a list of colours built with a list comprehension, `ax.barh()`, `ax.bar_label()`, `ax.xaxis.set_visible(False)`, `ax.spines[...]`.

## 9. Want more practice? (optional)

Nothing to hand in. If you want to try on your own, use `bike_share_mock.csv`:

1. **Which start station is busiest?** Count rides per start station and draw a sorted horizontal bar chart.
2. **Do casual riders take longer rides than members?** Compare ride durations for the two groups with a boxplot or two histograms, not a bar of averages. Decide what to do with the few very long rides, and say so in the title.

Write the question as the chart title before you write any code.